In [1]:
from daedalus.custom_instruments import daedalusProjFieldDAQ, DAQmxAdapter
%matplotlib notebook
import numpy as np
from collections import OrderedDict
import pandas as pd
import pymeasure
from time import sleep, time
from matplotlib import pyplot as plt
from matplotlib import style
from datetime import datetime
from pymeasure.instruments.keithley import Keithley2400
#from pymeasure.instruments.keithley import Keithley2182A
#from pymeasure.instruments.srs import SR570
import matplotlib.cm as cm
from glob import glob

ImportError: cannot import name 'RangeException' from 'pymeasure.instruments' (C:\Users\RalphGroup\anaconda3\lib\site-packages\pymeasure\instruments\__init__.py)

In [14]:
magnet = daedalusProjFieldDAQ(DAQmxAdapter("Dev1", ['ao0', 'ai4']),"GPIB::10")
sourcemeter = Keithley2400("GPIB::4")

KeyboardInterrupt: 

In [ ]:
def field_sweep_daedalus(Bmin,Bmax,Bstep,average,delay,file_path,sample_name, hysteresis=False):
    plt.show()
    BdataPoints= int((Bmax-Bmin)/Bstep)+1
    field=np.linspace(Bmin,Bmax,num=BdataPoints)
    
    if hysteresis:
        print('hysteresis sweep')
        field = field.tolist() + field.tolist()[-2::-1]
        
    data=OrderedDict()
    data['field']=[]
    data['V2p']=[]

    filename =file_path+'Field_'+sample_name+ str(Bmin) + '_' + str(Bmax) + '_' + str(Bstep)\
    + '_' +str(round(sourcemeter.source_current,5))+'uA_' + '.csv'
    
    fig = plt.figure()
    ax = fig.add_subplot(111)

    fig.show()
    fig.canvas.draw()


    magnet.field = Bmin
    print ('waiting for initial field')
    while abs(ppms.getField()[1]-Bmin)>1:
        sleep (3)
    print ('Intial field set at : ' + str(ppms.getField()[1]))
    
    for i in field:
        ppms.setField(i, 100)
        print ('Check Field :' + str(i))
        while abs(ppms.getField()[1]-i)>1:
            sleep (1)
        sleep(5)
        print ('Field : ' + str(ppms.getField()[1]) + ' checked and stabilized')
        
        Va=[] #points for average at one angle

        for a in range(average):
            Va.append(sourcemeter.voltage[0])

            sleep(delay*1)
            
        data['field'].append(ppms.getField()[1])
        data['V2p'].append(np.mean(Va))
        data['Temp'].append(ppms.getTemperature()[1])

        ax.clear()
        ax.plot(data['field'], data['V2p'])
        fig.canvas.draw()
        ax.relim()
        ax.autoscale_view(True,True,True)

        plt.xlabel('Field (Oe)',fontsize=18)
        plt.ylabel('V (V)',fontsize=18)
    
        print ('Measurement done for Field : ' + str(ppms.getField()[1]))
        dataset = pd.DataFrame({ 'Field': np.transpose(data['field']), 'V2p':np.transpose(data['V2p']), 'Temperature':np.transpose(data['Temp'])})

#save dataframe to csv file in the same path and file name
        dataset.to_csv(filename,index=False)
    plt.ioff()
    plt.show()

    
    #save data to dataframe
    dataset = pd.DataFrame({'Field': np.transpose(data['field']), 'V2p':np.transpose(data['V2p']), 'Temperature':np.transpose(data['Temp'])})
    
    #save dataframe to csv file in the same path and file name
    dataset.to_csv(filename,index=False)
    